# Lab 21 — BONUS ALL (B1–B5)

Chọn GPU runtime, thêm `HF_TOKEN` và `GITHUB_TOKEN` trong Colab Secrets, rồi chạy tuần tự. Các stage train có resume.

In [ ]:
# @title Setup
import os, pathlib, runpy, subprocess, sys, torch
REPO = "https://github.com/ashura102938475/Day21-Track3-Finetuning-Lab.git"
WORK = pathlib.Path("/content/Day21-Track3-Finetuning-Lab")
if not WORK.exists(): subprocess.run(["git", "clone", REPO, str(WORK)], check=True)
os.chdir(WORK)
subprocess.run(["git", "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if not torch.cuda.is_available(): raise RuntimeError("Chọn Runtime > Change runtime type > GPU")
# Match the already-frozen core model; Colab GPU is hardware, not a reason to change the experiment.
os.environ["COMPUTE_TIER"] = "LAPTOP"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
_bonus = runpy.run_path("notebooks/07_bonus_all.py", run_name="lab21_bonus")
_stages = {"b1": _bonus["stage_b1"], "b3": _bonus["stage_b3"], "b4": _bonus["stage_b4"]}
def run_stage(name):
    print(f"Running {name} in the Colab kernel; exceptions below are the real root cause.")
    return _stages[name]()
print("GPU ready:", torch.cuda.get_device_name(0))

In [ ]:
# @title Secrets
import os
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
if not HF_TOKEN or not GITHUB_TOKEN: raise RuntimeError("Add HF_TOKEN and GITHUB_TOKEN to Colab Secrets")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
print("Secrets loaded (values hidden)")

In [ ]:
# @title B2 — custom + reasoning datasets
from scripts import bonus_data, bonus_verify
summary = bonus_data.write_all()
print("dataset rows:", summary["n"], "schema errors:", summary["schema_errors"])
if bonus_verify.main(["--allow-unrun-gpu"]):
    raise RuntimeError("Bonus data verification failed; read the FAIL line above.")

In [ ]:
# @title B1 — merge + hot-swap
run_stage("b1")

In [ ]:
# @title B3 — reasoning-trace collapse (resumable)
run_stage("b3")

In [ ]:
# @title B4 — controlled rank sweep (resumable)
run_stage("b4")

In [ ]:
# @title Verify first, then B5 publish
import os, subprocess
from scripts import bonus_verify, publish_bonus, verify as core_verify

def require_ok(label, code):
    if code:
        raise RuntimeError(f"{label} failed; read the FAIL line above.")

def git(*args, check=True, env=None):
    result = subprocess.run(["git", *args], text=True, capture_output=True, env=env)
    output = (result.stdout or "") + (result.stderr or "")
    if output.strip(): print(output.rstrip())
    if check and result.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed with exit {result.returncode}\n{output}")
    return result

require_ok("Core verification", core_verify.main([]))
require_ok("Bonus prerequisites", bonus_verify.main(["--allow-unrun-gpu"]))
require_ok("Hugging Face publication", publish_bonus.main())
require_ok("Final bonus verification", bonus_verify.main([]))
paths = ["data/CUSTOM_DATASET.md", "data/train_custom_ai_course.jsonl", "data/train_reasoning_trace.jsonl", "data/trace_holdout.jsonl", "results/merge_check.json", "results/bonus", "submission/REPORT.md", "LINKS.md"]
git("add", "-f", "--", *paths)
commit_env = {**os.environ, "GIT_AUTHOR_NAME": "NGUYỄN ANH TRÀ", "GIT_AUTHOR_EMAIL": "tra01020407@gmail.com", "GIT_COMMITTER_NAME": "NGUYỄN ANH TRÀ", "GIT_COMMITTER_EMAIL": "tra01020407@gmail.com"}
if git("diff", "--cached", "--quiet", check=False).returncode:
    git("commit", "-m", "Complete Lab 21 bonuses B1-B5", env=commit_env)
    if git("remote", "get-url", "mine", check=False).returncode:
        git("remote", "add", "mine", REPO)
    else:
        git("remote", "set-url", "mine", REPO)
    helper = '!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f'
    git("-c", "credential.helper=" + helper, "push", "mine", "HEAD:main", env=os.environ.copy())
print("All bonus evidence published")